# Lab 4: Tool calling and structured output

## Overview

A chat model that only returns prose is hard to build on. Two features make it programmable. **Structured output** forces the reply to match a JSON schema you supply, so your code can parse it without guesswork. **Tool calling** lets the model ask your code to run a function, see the result, and continue. Together they are how most agents and assistants are built.

Both follow the OpenAI API, so the code in this lab is what you would write against any compatible provider.

## Learning objectives

By the end of this lab you will be able to:

- Get a reply that conforms to a JSON schema and parse it safely
- Build the schema from a Pydantic model instead of writing it by hand
- Define tools, let the model choose one, run it, and return the result
- Force or forbid tool use with `tool_choice`
- Handle multiple tool calls in one turn

## Prerequisites

- Lab 1 completed, including the `.env` file
- About 25 minutes

## Step 1: Set up your environment

In [ ]:
# Installs the packages this lab needs. Safe to rerun.
%pip install -q "openai>=3,<4" python-dotenv pydantic

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv
from openai import OpenAI

# Reads OPENAI_BASE_URL and OPENAI_API_KEY from the .env file in the repository root.
# override=True makes the file win over any OPENAI_API_KEY already set in your shell.
load_dotenv(Path("..") / ".env", override=True)
load_dotenv(override=True)

base_url = os.environ["OPENAI_BASE_URL"]
api_key = os.environ["OPENAI_API_KEY"]
print("Endpoint:", base_url)
print("Key:     ", api_key[:6] + "..." + api_key[-4:])

client = OpenAI()  # The SDK reads the same two variables, so no arguments are needed.

## The call in one look

Both features are extra fields on the same chat completions call from Lab 1.

| Field | What it does | Shape |
|---|---|---|
| `response_format` | forces the reply to match a JSON schema | `{"type": "json_schema", "json_schema": {"name": "x", "schema": {...}}}` |
| `tools` | functions the model may ask you to run | list of `{"type": "function", "function": {"name", "description", "parameters"}}` |
| `tool_choice` | whether tools may be used | `"auto"` (default) or `{"type": "function", "function": {"name": "..."}}`; to forbid tools, omit `tools` |

With tools, check `choices[0].message.tool_calls`. Each has `id`, `function.name`, and `function.arguments` (a JSON string). You run the function and reply with `{"role": "tool", "tool_call_id": id, "content": result}`. The minimal structured call:

In [ ]:
import json

response = client.chat.completions.create(
    model="hackathon-chat",
    messages=[{"role": "user", "content": "Give me a color and its hex code."}],
    response_format={"type": "json_schema", "json_schema": {"name": "color", "schema": {
        "type": "object",
        "properties": {"name": {"type": "string"}, "hex": {"type": "string"}},
        "required": ["name", "hex"], "additionalProperties": False}}},
    max_tokens=60,
)
print(json.loads(response.choices[0].message.content))

## Step 2: Ask for JSON the naive way

Start with what most people try first: ask for JSON in the prompt. Sometimes it works. Often the model adds a sentence, wraps the JSON in a code fence, or renames a field. Your parser breaks at 2 AM.

In [ ]:
import json

response = client.chat.completions.create(
    model="hackathon-chat",
    messages=[{"role": "user", "content": "Extract the event details as JSON with keys name, city, and date: "
               "HackMidwest runs in Kansas City on October 11, 2026."}],
    max_tokens=120,
)
text = response.choices[0].message.content
print(text)

try:
    print("\nParsed:", json.loads(text))
except json.JSONDecodeError as e:
    print("\njson.loads failed:", e)

## Step 3: Constrain the output with a JSON schema

With `response_format` set to `json_schema`, the server constrains generation token by token so the reply cannot leave the schema. No fences, no extra words, every required field present.

In [ ]:
schema = {
    "type": "object",
    "properties": {
        "name": {"type": "string"},
        "city": {"type": "string"},
        "date": {"type": "string", "description": "ISO 8601 date"},
    },
    "required": ["name", "city", "date"],
    "additionalProperties": False,
}

response = client.chat.completions.create(
    model="hackathon-chat",
    messages=[{"role": "user", "content": "Extract the event details: HackMidwest runs in Kansas City on October 11, 2026."}],
    response_format={"type": "json_schema", "json_schema": {"name": "event", "schema": schema}},
    max_tokens=120,
)
text = response.choices[0].message.content
print(text)
event = json.loads(text)  # Safe now.
print("\nCity:", event["city"])

Writing schemas by hand gets old. Define a Pydantic model, let it produce the schema, and validate the reply with the same model. One definition, both directions.

In [ ]:
from pydantic import BaseModel, Field

class Idea(BaseModel):
    title: str = Field(description="Short project name")
    pitch: str = Field(description="One sentence")
    difficulty: int = Field(ge=1, le=5)
    apis_used: list[str]

class Ideas(BaseModel):
    ideas: list[Idea]

response = client.chat.completions.create(
    model="hackathon-chat",
    messages=[{"role": "user", "content": "Suggest two hackathon project ideas that use a chat model and an image model."}],
    response_format={"type": "json_schema", "json_schema": {"name": "ideas", "schema": Ideas.model_json_schema()}},
    max_tokens=400,
)
ideas = Ideas.model_validate_json(response.choices[0].message.content)
for idea in ideas.ideas:
    print(f"{idea.title} (difficulty {idea.difficulty}): {idea.pitch}")

> NOTE: A schema guarantees shape, not truth. The model can still fill a field with a wrong value. Validate anything that matters.

## Step 4: Give the model a tool

A tool is a function you describe to the model: its name, what it does, and its parameters as a JSON schema. The model never runs it. When it decides the function is needed, it replies with a `tool_calls` entry containing the name and arguments, and stops. Your code runs the function and sends the result back.

The function below is a stand-in for anything real: a database query, an HTTP call, a calculation.

In [ ]:
# Fake data so the lab has no external dependencies. Replace with real lookups in your project.
SCHEDULE = {
    "opening": "Saturday 9:00 AM, main hall",
    "submissions": "Sunday 12:00 PM, via Devpost",
    "judging": "Sunday 1:00 PM to 3:30 PM",
    "closing": "Sunday 4:00 PM, main hall",
}

def get_schedule(event: str) -> str:
    return SCHEDULE.get(event.lower(), f"No schedule entry for '{event}'")

tools = [{
    "type": "function",
    "function": {
        "name": "get_schedule",
        "description": "Look up when a hackathon event happens. Events: opening, submissions, judging, closing.",
        "parameters": {
            "type": "object",
            "properties": {"event": {"type": "string", "description": "Event name"}},
            "required": ["event"],
        },
    },
}]

messages = [{"role": "user", "content": "When is judging?"}]
response = client.chat.completions.create(model="hackathon-chat", messages=messages, tools=tools, max_tokens=200)

message = response.choices[0].message
print("Finish reason:", response.choices[0].finish_reason)
print("Content:     ", repr(message.content))
print("Tool calls:  ", message.tool_calls)

`finish_reason` is `tool_calls`, `content` is empty, and `tool_calls` names the function with its arguments as a JSON string. Now run it and send the result back in a `tool` message. The model then writes the answer.

In [ ]:
call = message.tool_calls[0]
args = json.loads(call.function.arguments)
result = get_schedule(**args)
print(f"Ran {call.function.name}({args}) -> {result!r}")

messages.append(message)  # The assistant turn that requested the tool.
messages.append({"role": "tool", "tool_call_id": call.id, "content": result})

final = client.chat.completions.create(model="hackathon-chat", messages=messages, tools=tools, max_tokens=200)
print("\nAnswer:", final.choices[0].message.content)

## Step 5: Write the loop once

Real applications do not know in advance how many tool calls a request needs. The pattern is a loop: call the model, run every tool it asked for, append the results, repeat until it answers in prose. Cap the iterations so a confused model cannot spin forever.

In [ ]:
TOOLS = {"get_schedule": get_schedule}

def run(question, max_steps=5):
    messages = [{"role": "user", "content": question}]
    for step in range(max_steps):
        response = client.chat.completions.create(model="hackathon-chat", messages=messages, tools=tools, max_tokens=300)
        message = response.choices[0].message
        messages.append(message)
        if not message.tool_calls:
            return message.content
        for call in message.tool_calls:  # The model may request several at once.
            args = json.loads(call.function.arguments)
            result = TOOLS[call.function.name](**args)
            print(f"  step {step + 1}: {call.function.name}({args}) -> {result!r}")
            messages.append({"role": "tool", "tool_call_id": call.id, "content": result})
    return "Stopped after too many steps."

print(run("When do submissions close, and when is the closing ceremony?"))

Two tool calls, one turn, and the answer combines both results. Everything your tools return goes back into the context window, so keep results short.

## Step 6: Control when tools are used

`tool_choice` decides whether the model may or must call a tool. The default `auto` is right for assistants. Force a specific function when you are using the model as a parser. To forbid tools for a request, leave `tools` out of that request.

> NOTE: The OpenAI API also defines `tool_choice="none"`. On this deployment it is unreliable: the model may still write a tool call, the server discards it, and you get an empty message. Omit `tools` instead.

In [ ]:
# Force the call even though the question does not obviously need it.
forced = client.chat.completions.create(
    model="hackathon-chat",
    messages=[{"role": "user", "content": "Tell me about the opening."}],
    tools=tools,
    tool_choice={"type": "function", "function": {"name": "get_schedule"}},
    max_tokens=100,
)
print("Forced:", forced.choices[0].message.tool_calls[0].function.arguments)

# No tools in the request: the model answers from what it knows, which here is nothing useful.
plain = client.chat.completions.create(
    model="hackathon-chat",
    messages=[{"role": "user", "content": "When is judging at this hackathon? One sentence."}],
    max_tokens=60,
)
print("No tools:", plain.choices[0].message.content)

## Things to know

- **Arguments are model output.** Validate them before you act, exactly as you would validate user input. Never pass them to a shell or a database unchecked.
- **Keep tool descriptions specific.** The description is the only thing the model reads to decide. "Look up when a hackathon event happens" works; "schedule helper" does not.
- **Streaming works with tools.** `tool_calls` arrive as deltas you accumulate by index. Start without streaming until the loop works.
- **Thinking and tools combine.** With `enable_thinking` on, the reasoning arrives in the `reasoning` field before any tool call, at the time cost from Lab 1.
- **Structured output and tools together.** Use tools to gather facts, then a final call with `response_format` to shape the answer.

## Validation

In [ ]:
check = client.chat.completions.create(
    model="hackathon-chat",
    messages=[{"role": "user", "content": "Return the number forty-two."}],
    response_format={"type": "json_schema", "json_schema": {"name": "n", "schema": {
        "type": "object", "properties": {"value": {"type": "integer"}}, "required": ["value"], "additionalProperties": False}}},
    max_tokens=20,
)
assert json.loads(check.choices[0].message.content)["value"] == 42
assert message.tool_calls and message.tool_calls[0].function.name == "get_schedule"
print("Lab 4 complete.")

## Clean up

Nothing to clean up.

## Next steps

You have the full toolkit: chat, retrieval, images, structured output, and tools. A typical hackathon project wires them in that order: retrieve facts, let the model call tools for live data, and return a structured result your frontend renders. Build the smallest version that works end to end first, then improve one piece at a time.